# CycloSafe Detection V2 Training
Run this notebook in Google Colab to train the Cyclone Detection V2 model.

In [ ]:
# STEP 1 - Install/check dependencies
!pip install tensorflow opencv-python scikit-learn matplotlib requests
import tensorflow as tf
print('TensorFlow version:', tf.__version__)

In [ ]:
# STEP 2 - Check GPU
gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    print('GPU Information:', tf.config.experimental.get_device_details(gpu_devices[0]))
else:
    print('WARNING: No GPU detected. Training will be very slow! Go to Runtime > Change runtime type and select GPU.')

In [ ]:
# STEP 3 - Upload or mount the existing dataset_v2
from google.colab import drive
import os
import glob
drive.mount('/content/drive')

# UPDATE THIS PATH to point to your dataset_v2 folder in Google Drive.
# For example, if you uploaded the backend folder to the root of your Drive:
BASE_DIR = '/content/drive/MyDrive/backend/ml'
DATASET_DIR = os.path.join(BASE_DIR, 'dataset_v2')
MODELS_DIR = os.path.join(BASE_DIR, 'models')

os.makedirs(MODELS_DIR, exist_ok=True)
print('\nDataset path:', DATASET_DIR)

cyclone_files = glob.glob(os.path.join(DATASET_DIR, 'Cyclone', '*.*'))
no_cyclone_files = glob.glob(os.path.join(DATASET_DIR, 'No_Cyclone', '*.*'))
print('Cyclone image count:', len(cyclone_files))
print('No_Cyclone image count:', len(no_cyclone_files))

if len(cyclone_files) == 0 or len(no_cyclone_files) == 0:
    print('\nERROR: Dataset not found or empty. Please check the BASE_DIR path above.')

In [ ]:
# STEP 4 - Load the existing Detection V2 training code
import json
import numpy as np
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve
import cv2
from collections import defaultdict
import matplotlib.pyplot as plt

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
V2_MODEL_PATH = os.path.join(MODELS_DIR, 'cyclone_detector_v2.keras')
CONFIG_PATH = os.path.join(MODELS_DIR, 'detection_v2_config.json')

def get_event_id(filename, class_name):
    basename = os.path.basename(filename)
    if class_name == 'Cyclone':
        if basename.startswith('cyclone_'):
            parts = basename.split('_')
            if len(parts) >= 3:
                return parts[1]
        if len(basename) >= 6 and basename[:6].isdigit():
            return basename[:6]
        return 'unknown_cyclone'
    else:
        return basename

def prepare_data():
    classes = ['No_Cyclone', 'Cyclone']
    events = defaultdict(list)
    for label, class_name in enumerate(classes):
        class_dir = os.path.join(DATASET_DIR, class_name)
        if not os.path.exists(class_dir): continue
        for fname in os.listdir(class_dir):
            if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                fpath = os.path.join(class_dir, fname)
                evt_id = get_event_id(fpath, class_name)
                events[(label, evt_id)].append(fpath)
    train_paths, train_labels = [], []
    val_paths, val_labels = [], []
    test_paths, test_labels = [], []
    for label in [0, 1]:
        label_events = [k for k in events.keys() if k[0] == label]
        np.random.seed(42)
        np.random.shuffle(label_events)
        total_images = sum(len(events[k]) for k in label_events)
        train_target = int(0.70 * total_images)
        val_target = int(0.15 * total_images)
        train_cnt, val_cnt = 0, 0
        for k in label_events:
            group = events[k]
            if train_cnt < train_target:
                train_paths.extend(group); train_labels.extend([label]*len(group)); train_cnt += len(group)
            elif val_cnt < val_target:
                val_paths.extend(group); val_labels.extend([label]*len(group)); val_cnt += len(group)
            else:
                test_paths.extend(group); test_labels.extend([label]*len(group))
    print(f'TRAIN count: {len(train_labels)} (Cyclone = {train_labels.count(1)}, No_Cyclone = {train_labels.count(0)})')
    print(f'VALIDATION count: {len(val_labels)} (Cyclone = {val_labels.count(1)}, No_Cyclone = {val_labels.count(0)})')
    print(f'TEST count: {len(test_labels)} (Cyclone = {test_labels.count(1)}, No_Cyclone = {test_labels.count(0)})')
    return (train_paths, train_labels), (val_paths, val_labels), (test_paths, test_labels)

class DataGenerator(tf.keras.utils.Sequence):
    def __init__(self, paths, labels, batch_size=32, augment=False, shuffle=True):
        self.paths, self.labels, self.batch_size, self.augment, self.shuffle = paths, labels, batch_size, augment, shuffle
        self.indexes = np.arange(len(self.paths))
        if self.shuffle: np.random.shuffle(self.indexes)
        self.aug_model = tf.keras.Sequential([tf.keras.layers.RandomFlip('horizontal_and_vertical'), tf.keras.layers.RandomRotation(0.2), tf.keras.layers.RandomZoom(0.1)]) if augment else None
    def __len__(self): return int(np.ceil(len(self.paths) / self.batch_size))
    def on_epoch_end(self): 
        if self.shuffle: np.random.shuffle(self.indexes)
    def __getitem__(self, index):
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        batch_paths = [self.paths[k] for k in batch_indexes]
        batch_labels = [self.labels[k] for k in batch_indexes]
        X = np.empty((len(batch_paths), *IMG_SIZE, 3), dtype=np.float32)
        for i, p in enumerate(batch_paths):
            img = cv2.imread(p)
            if img is not None:
                X[i,] = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), IMG_SIZE)
        y = np.array(batch_labels, dtype=np.float32)
        if self.augment:
            X = self.aug_model(X, training=True).numpy()
        return X, y

def build_model():
    base_model = EfficientNetV2B0(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
    base_model.trainable = False
    inputs = Input(shape=(*IMG_SIZE, 3))
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.3)(x)
    outputs = Dense(1, activation='sigmoid')(x)
    return Model(inputs, outputs), base_model

print('Training code loaded.')

In [ ]:
# STEP 5 - Train the model
train_data, val_data, test_data = prepare_data()

train_gen = DataGenerator(train_data[0], train_data[1], batch_size=BATCH_SIZE, augment=True, shuffle=True)
val_gen = DataGenerator(val_data[0], val_data[1], batch_size=BATCH_SIZE, augment=False, shuffle=False)
test_gen = DataGenerator(test_data[0], test_data[1], batch_size=BATCH_SIZE, augment=False, shuffle=False)

total = len(train_data[1])
pos = sum(train_data[1])
neg = total - pos
class_weight = {0: (1 / neg) * (total / 2.0), 1: (1 / pos) * (total / 2.0)}
print(f'Class Weights: {class_weight}')

model, base_model = build_model()

print('\n--- STAGE 1: Training Head ---')
model.compile(optimizer=Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
es1 = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
model.fit(train_gen, validation_data=val_gen, epochs=10, callbacks=[es1], class_weight=class_weight)

print('\n--- STAGE 2: Fine-tuning Backbone ---')
base_model.trainable = True
for layer in base_model.layers[:-30]: layer.trainable = False
model.compile(optimizer=Adam(1e-5), loss='binary_crossentropy', metrics=['accuracy'])

es2 = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
mc = ModelCheckpoint(V2_MODEL_PATH, monitor='val_loss', save_best_only=True)
rlr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7)
model.fit(train_gen, validation_data=val_gen, epochs=20, callbacks=[es2, mc, rlr], class_weight=class_weight)

print(f'\nModel saved to {V2_MODEL_PATH}')

In [ ]:
# STEP 6 & 8 & 9 - Evaluate on the existing test set and Save Config
print('Evaluating V2 on Validation set to find optimal threshold...')
model = load_model(V2_MODEL_PATH)
val_probs = model.predict(val_gen).flatten()
val_true = np.array(val_data[1])
fpr, tpr, thresholds = roc_curve(val_true, val_probs)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]
if optimal_threshold < 0.2 or optimal_threshold > 0.8: optimal_threshold = 0.5
print(f'Selected Optimal Threshold: {optimal_threshold}')

print('\nEvaluating V2 on Test set...')
test_probs_v2 = model.predict(test_gen).flatten()
test_true = np.array(test_data[1])
test_pred_v2 = (test_probs_v2 >= optimal_threshold).astype(int)

cm = confusion_matrix(test_true, test_pred_v2)
tn, fp, fn, tp = cm[0][0], cm[0][1], cm[1][0], cm[1][1]
print(f"Accuracy:  {accuracy_score(test_true, test_pred_v2):.4f}")
print(f"Precision: {precision_score(test_true, test_pred_v2, zero_division=0):.4f}")
print(f"Recall:    {recall_score(test_true, test_pred_v2, zero_division=0):.4f}")
print(f"F1:        {f1_score(test_true, test_pred_v2, zero_division=0):.4f}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")

# STEP 9 - Save detection_v2_config.json
config = {
    'input_size': [224, 224, 3],
    'class_names': ['No_Cyclone', 'Cyclone'],
    'classification_threshold': float(optimal_threshold),
    'training_dataset_info': {
        'total_train': len(train_data[1]),
        'total_val': len(val_data[1]),
        'total_test': len(test_data[1])
    },
    'model_architecture': 'EfficientNetV2B0',
    'preprocessing': 'Input normalized internally by EfficientNetV2B0 (expects 0-255 RGB)'
}
with open(CONFIG_PATH, 'w') as f:
    json.dump(config, f, indent=4)
print(f'\nConfig saved to {CONFIG_PATH}')

In [ ]:
# STEP 7 - Run challenge testing (OOD Images)
import requests
CHALLENGE_URLS = [
    'https://upload.wikimedia.org/wikipedia/commons/thumb/c/c5/M101_hires_STScI-PRC2006-10a.jpg/800px-M101_hires_STScI-PRC2006-10a.jpg',
    'https://upload.wikimedia.org/wikipedia/commons/thumb/c/c4/NGC_4414_%28NASA-med%29.jpg/800px-NGC_4414_%28NASA-med%29.jpg',
    'https://upload.wikimedia.org/wikipedia/commons/thumb/2/29/Thunderstorm_in_Ann_Arbor.jpg/800px-Thunderstorm_in_Ann_Arbor.jpg',
    'https://upload.wikimedia.org/wikipedia/commons/thumb/d/d4/Squall_line_approaching.jpg/800px-Squall_line_approaching.jpg'
]
images = []
urls = []
print('Downloading OOD Challenge Images...')
for url in CHALLENGE_URLS:
    try:
        r = requests.get(url)
        nparr = np.frombuffer(r.content, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        if img is not None:
            images.append(cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), (224, 224)))
            urls.append(url)
    except:
        pass

if images:
    print('\nEvaluating Challenge Set...')
    X_chal = np.array(images, dtype=np.float32)
    chal_probs = model.predict(X_chal).flatten()
    fp_chal = 0
    for i, p in enumerate(chal_probs):
        is_cyc = p >= optimal_threshold
        if is_cyc: fp_chal += 1
        name = urls[i].split('/')[-1]
        print(f'{name}: Prob={p:.4f} -> {\'CYCLONE (False Positive)\' if is_cyc else \'NO_CYCLONE (Correct)\'}')
    print(f'\nTotal Challenge False Positives: {fp_chal} / {len(urls)}')


In [ ]:
# STEP 10 - Provide downloadable model files
from google.colab import files
print('Ready to download the trained model and config to your local machine.')
print('If you mapped Google Drive, they are already saved in the models/ folder there.')
print('Alternatively, run the lines below to download them through the browser:')

# Uncomment to download directly to your browser
# files.download(V2_MODEL_PATH)
# files.download(CONFIG_PATH)